In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import sys
sys.path.append("data")

import pandas as pd
from functions import get_features_and_target, get_train_test_split

In [25]:
df = pd.read_csv("data/market_data.csv", parse_dates=["date"])
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df.to_csv("data/market_data.csv", index=False)
print(df.columns.tolist())

['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume']


In [26]:
df = pd.read_csv("data/market_data.csv", parse_dates=["date"])
print(df.shape)
df.head()

(339, 8)


,date,ticker,open,high,low,close,adj_close,volume
0,2026-02-02,AAPL,260.029999,270.489990,259.209991,270.010010,269.509308,73913400
1,2026-02-03,AAPL,269.200012,271.880005,267.609985,269.480011,268.980286,64394700
2,2026-02-04,AAPL,272.290009,278.950012,272.290009,276.489990,275.977234,90545700
3,2026-02-05,AAPL,278.130005,279.500000,273.230011,275.910004,275.398346,52977400
4,2026-02-06,AAPL,277.119995,280.910004,276.929993,278.119995,277.604218,50453400


In [27]:
df["ticker"].value_counts()

ticker
AAPL    113
BE      113
NKE     113
Name: count, dtype: int64

In [28]:
X, y, dates = get_features_and_target(df)
print(X.shape, y.shape, dates.shape)

(279, 6) (279,) (279,)


In [29]:
df.isna().sum()

date         0
ticker       0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

In [30]:
assert not set(["date", "ticker", "open", "high", "low", "close", "adj_close"]) & set(X.columns)

assert X.isna().sum().sum() == 0
assert y.isna().sum() == 0

y.value_counts()

target
0    147
1    132
Name: count, dtype: int64

In [31]:
X_train, y_train, X_test, y_test = get_train_test_split(X, y, dates)
print(len(X_train), len(X_test))
print("Split is chronological — verified inside get_train_test_split.")

222 57
Split is chronological — verified inside get_train_test_split.


In [32]:
print("X_train:", len(X_train), " X_test:", len(X_test))
print("Function-internal chronological check already passed — no overlap between train and test dates.")

X_train: 222  X_test: 57
Function-internal chronological check already passed — no overlap between train and test dates.


In [33]:
from training import train_logistic_regression, train_random_forest, evaluate

In [34]:
log_reg, log_reg_preds = train_logistic_regression(X_train, y_train, X_test, y_test)
evaluate("Logistic Regression", y_test, log_reg_preds)


=== Logistic Regression ===
Accuracy:  0.4386
Precision: 0.3947
Recall:    0.6250
F1 score:  0.4839
Confusion matrix (rows=actual, cols=predicted):
[[10 23]
 [ 9 15]]
              precision    recall  f1-score   support

           0       0.53      0.30      0.38        33
           1       0.39      0.62      0.48        24

    accuracy                           0.44        57
   macro avg       0.46      0.46      0.43        57
weighted avg       0.47      0.44      0.43        57



In [35]:
rf, rf_preds = train_random_forest(X_train, y_train, X_test, y_test)
evaluate("Random Forest", y_test, rf_preds)


=== Random Forest ===
Accuracy:  0.4561
Precision: 0.4186
Recall:    0.7500
F1 score:  0.5373
Confusion matrix (rows=actual, cols=predicted):
[[ 8 25]
 [ 6 18]]
              precision    recall  f1-score   support

           0       0.57      0.24      0.34        33
           1       0.42      0.75      0.54        24

    accuracy                           0.46        57
   macro avg       0.50      0.50      0.44        57
weighted avg       0.51      0.46      0.42        57

